# Chapter 3 — Multimodal Fusion

This notebook builds the **joint multimodal input sequence** for nanochat_vlm.

Goal:
    [image patches] + [text tokens] → single LM input

This notebook:
- prepends image tokens
- constructs embeddings
- defines attention masks

No training happens here.

## Fusion Strategy: Prepend Image Tokens

We use a **prepend strategy**:

    [<im_start>, P1, P2, ..., Pn, <im_end>] + text

Reasons:
- simple
- widely used (LLaVA-style)
- causal masking remains valid

## Symbol-Level Sequence

Token sequence (conceptual):

    <bos>
    <im_start>
    <im_patch> × N
    <im_end>
    text tokens...

In [ ]:
import torch

## Dimensions

These must match previous notebooks.

In [ ]:
BATCH = 2
NUM_PATCHES = 256
LM_DIM = 2048
TEXT_LEN = 16

## Inputs

We assume:
- projected vision embeddings are ready
- text token embeddings are ready

In [ ]:
# Vision (after projector)
vision_embeds = torch.randn(BATCH, NUM_PATCHES, LM_DIM)

# Text token embeddings (from LM embedding table)
text_embeds = torch.randn(BATCH, TEXT_LEN, LM_DIM)

## Special Token Embeddings

<im_start> and <im_end> are learned embeddings
from the LM embedding table.

In [ ]:
im_start_embed = torch.randn(1, 1, LM_DIM)
im_end_embed   = torch.randn(1, 1, LM_DIM)

im_start_embed = im_start_embed.expand(BATCH, -1, -1)
im_end_embed   = im_end_embed.expand(BATCH, -1, -1)

## Image Block Construction

Final image block shape:
    (B, 1 + N + 1, D)

In [ ]:
image_block = torch.cat(
    [im_start_embed, vision_embeds, im_end_embed],
    dim=1
)

print("Image block shape:", image_block.shape)

## Joint Multimodal Sequence

Final sequence:
    [image block] + [text]

In [ ]:
joint_embeds = torch.cat(
    [image_block, text_embeds],
    dim=1
)

print("Joint sequence shape:", joint_embeds.shape)

## Attention Masking

We must enforce:
- no token attends to the future
- image tokens attend freely within image block
- text tokens attend to all image tokens

## Causal Mask (Concept)

Let L = total sequence length.

Mask:
    mask[i, j] = 1 if j ≤ i
                 0 otherwise


---

## 💻 Code Block 7 — Build Causal Mask

```python
L = joint_embeds.shape[1]

attn_mask = torch.tril(torch.ones(L, L)).bool()
attn_mask = attn_mask.unsqueeze(0).expand(BATCH, -1, -1)

print("Attention mask shape:", attn_mask.shape)


## Optional: Explicit Region Awareness

Some models track:
- image token range
- text token range

Useful for debugging or loss masking.

In [ ]:
image_len = image_block.shape[1]
text_start = image_len

print("Image tokens:", image_len)
print("Text starts at index:", text_start)

## Final Contract

Inputs to LM:
- joint_embeds: (B, L, D)
- attn_mask:    (B, L, L)

This is the **only multimodal interface** the LM sees.

## Next Steps

With fusion defined, we can now:
- plug into nanochat transformer
- compute logits
- train end-to-end

Next notebook:
    Chapter 4 — Multimodal Forward Pass